# objective contract v1

目标：验证 `src/objective` 替代旧 reward 概念，提供 label、MSE/classification/IC/ranking 评估和收益风险晋升评分。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.objective import (
    OBJECTIVE_NAMES,
    PromotionThresholds,
    build_forward_return_labels,
    build_promotion_scores,
    evaluate_prediction_objectives,
    write_objective_label_manifest,
)

EXPERIMENT = "objective_contract_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

In [2]:
dates = pd.date_range("2026-01-01", periods=5, freq="D")
prices = pd.DataFrame(
    {
        "date": list(dates) * 2,
        "instrument": ["SOXL"] * 5 + ["SOXX"] * 5,
        "close": [100.0, 102.0, 101.0, 104.0, 106.0, 50.0, 51.0, 50.5, 51.5, 52.0],
    }
)
labels = build_forward_return_labels(prices, horizons=(1, 2))
label_manifest = write_objective_label_manifest(
    CARD,
    labels,
    artifact_id=EXPERIMENT,
    horizons=(1, 2),
    labels_filename="objective_labels.csv",
)
labels.head(8)

,date,instrument,close,forward_return_1d,label_available_1d,classification_label_1d,ranking_label_1d,forward_return_2d,label_available_2d,classification_label_2d,ranking_label_2d
0,2026-01-01,SOXL,100.0,0.020000,True,1,1.0,0.010000,True,1,1.0
1,2026-01-02,SOXL,102.0,-0.009804,True,0,1.0,0.019608,True,1,1.0
2,2026-01-03,SOXL,101.0,0.029703,True,1,1.0,0.049505,True,1,1.0
3,2026-01-04,SOXL,104.0,0.019231,True,1,1.0,NaN,False,<NA>,NaN
4,2026-01-05,SOXL,106.0,NaN,False,<NA>,NaN,NaN,False,<NA>,NaN
5,2026-01-01,SOXX,50.0,0.020000,True,1,2.0,0.010000,True,1,2.0
6,2026-01-02,SOXX,51.0,-0.009804,True,0,2.0,0.009804,True,1,2.0
7,2026-01-03,SOXX,50.5,0.019802,True,1,2.0,0.029703,True,1,2.0


In [3]:
evaluation_frame = labels.loc[labels["label_available_1d"]].copy()
evaluation_frame["prediction"] = [0.03, -0.01, 0.02, 0.015, 0.02, -0.02, 0.01, 0.008]
evaluation_frame["target_return"] = evaluation_frame["forward_return_1d"]
objective_scores = evaluate_prediction_objectives(evaluation_frame, group_col="date")
pd.DataFrame([objective_scores]).to_csv(CARD / "objective_scores.csv", index=False)
objective_scores

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


{'schema_version': 'objective.evaluation.v1',
 'row_count': 8,
 'mse': 5.188051298619e-05,
 'classification_accuracy': 1.0,
 'ic': 0.9100124716106859,
 'rank_ic': 0.9999999999999999,
 'rank_group_count': 2}

In [4]:
model_summary = pd.DataFrame(
    {
        "model": ["ridge_mse", "gcn_mse", "fragile_rank"],
        "return_pct": [14.0, 21.0, 28.0],
        "relative_return_pct": [3.0, 7.5, 11.0],
        "sharpe": [1.1, 1.3, 0.9],
        "max_drawdown_pct": [9.0, 13.0, 31.0],
        "turnover": [0.9, 1.4, 3.8],
    }
)
thresholds = PromotionThresholds(max_drawdown_pct=18.0, max_turnover=2.0, min_relative_return_pct=0.0)
promotion_scores = build_promotion_scores(model_summary, thresholds=thresholds)
promotion_scores.to_csv(CARD / "promotion_scores.csv", index=False)
promotion_scores

,schema_version,rank,model,promotion_score,promotion_decision,promotion_blockers,return_pct,relative_return_pct,sharpe,max_drawdown_pct,turnover,thresholds
0,objective.promotion_score.v1,1,gcn_mse,14.95,promote_candidate,,21.0,7.5,1.3,13.0,1.4,"{'min_return_pct': None, 'min_relative_return_..."
1,objective.promotion_score.v1,2,ridge_mse,11.95,promote_candidate,,14.0,3.0,1.1,9.0,0.9,"{'min_return_pct': None, 'min_relative_return_..."
2,objective.promotion_score.v1,3,fragile_rank,-9.35,blocked,"max_drawdown_pct,turnover",28.0,11.0,0.9,31.0,3.8,"{'min_return_pct': None, 'min_relative_return_..."


In [5]:
summary = {
    "schema_version": "research.objective_contract_v1",
    "status": "completed",
    "objective_names": list(OBJECTIVE_NAMES),
    "label_rows": int(len(labels)),
    "label_manifest": label_manifest,
    "evaluation": objective_scores,
    "promotion_rows": int(len(promotion_scores)),
    "promotion_candidates": promotion_scores.loc[
        promotion_scores["promotion_decision"] == "promote_candidate", "model"
    ].tolist(),
    "blocked_models": promotion_scores.loc[promotion_scores["promotion_decision"] == "blocked", "model"].tolist(),
    "artifacts": {
        "objective_labels": "objective_labels.csv",
        "objective_label_manifest": "objective_label_manifest.json",
        "objective_scores": "objective_scores.csv",
        "promotion_scores": "promotion_scores.csv",
    },
}
(CARD / "analysis_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
summary

{'schema_version': 'research.objective_contract_v1',
 'status': 'completed',
 'objective_names': ['classification', 'mse', 'ic', 'ranking', 'utility'],
 'label_rows': 10,
 'label_manifest': {'schema_version': 'objective.label_manifest.v1',
  'artifact_id': 'objective_contract_v1',
  'artifact_type': 'objective_labels',
  'producer': 'src.objective.specs',
  'paths': {'labels': 'objective_labels.csv',
   'manifest': 'objective_label_manifest.json'},
  'metadata': {'row_count': 10,
   'column_count': 11,
   'date_start': '2026-01-01',
   'date_end': '2026-01-05',
   'instrument_count': 2,
   'horizons': [1, 2],
   'label_columns': {'1d': {'forward_return': 'forward_return_1d',
     'classification': 'classification_label_1d',
     'ranking': 'ranking_label_1d',
     'available': 'label_available_1d'},
    '2d': {'forward_return': 'forward_return_2d',
     'classification': 'classification_label_2d',
     'ranking': 'ranking_label_2d',
     'available': 'label_available_2d'}},
   'label_a

## 中文结论

- 做了什么：新增并验证 `src/objective` v1 稳定接口，用 objective 替代 reward，覆盖 forward return label、classification label、ranking label、MSE/classification/IC/ranking 评估和晋升评分。
- 核心指标：本实验生成 10 行 label、1 组 objective score、3 个模型 promotion score；sidecar 为 `objective_labels.csv`、`objective_label_manifest.json`、`objective_scores.csv`、`promotion_scores.csv` 和 `analysis_summary.json`。
- regime 差异：本 notebook 不做 regime 切片；regime/window 差异仍由 `src/analysis` 消费 backtest 或 qlib recorder 结果完成。
- 是否晋升：`src/objective` 晋升为框架基础契约；样例里的模型排序不是交易模型晋升结论。
- 风险：label 使用未来价格生成，只能在训练/验证切分内部使用；生产推理侧不能读取未来 label。Promotion hard gate 只是默认框架，真实阈值需要在 production notebook 固化。
- 下一步：让模型训练和 backtest pipeline 默认写出 objective score 与 promotion score，并把它们接入 dashboard manifest。